# Ingesta TPC-H: lineitem incremental a Bronze

El modo se configura en `jobs/silver_config.yml`: `ingestion.mode: incremental`.
Se cargan solo claves nuevas. Filas identicas ya guardadas se omiten; cambios en claves
existentes y duplicados conflictivos se rechazan. La auditoria anterior se conserva.
Source y Bronze se leen para comparar; el timestamp de ingesta no es CDC del ERP.
Parquet append requiere un solo escritor, sin lectores concurrentes: no es una transaccion Delta.
Cerrar este kernel al terminar y abrir Silver con un kernel nuevo.


Source debe existir desde la generacion inicial de `01_ingesta_tpch.ipynb`. No se regeneran los archivos.


In [ ]:
from pathlib import Path
import sys
import yaml
from pyspark.sql import SparkSession

root = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "jobs" / "silver_config.yml").is_file()), None)
if root is None:
    raise FileNotFoundError("Ejecuta desde el proyecto o notebooks/.")
sys.path.insert(0, str(root / "jobs"))
from bronze_ingestion import BronzeIngestor

config_path = root / "jobs" / "silver_config.yml"
config = yaml.safe_load(config_path.read_text(encoding="utf-8"))
TABLE = "lineitem"
# None usa ingestion.mode del YAML; 'snapshot' reemplaza la tabla completa.
LOAD_MODE = None
source_path = config["ingestion"]["source_paths"][TABLE]
print(f"{TABLE}: {source_path} -> {config['sources'][TABLE]['path']}")


## Sesion Spark


In [ ]:
spark = (SparkSession.builder
    .appName(f"TPCH_Ingestion_{TABLE}_to_Bronze")
    .master(config["spark"]["master"])
    .config("spark.executor.memory", config["spark"]["configs"]["spark.executor.memory"])
    .config("spark.cores.max", config["spark"]["configs"]["spark.cores.max"])
    .config("spark.sql.shuffle.partitions", config["spark"]["configs"]["spark.sql.shuffle.partitions"])
    .config("spark.sql.session.timeZone", "UTC")
    .getOrCreate())


## Ejecutar ingesta y liberar recursos
El reporte distingue filas nuevas, existentes y duplicados exactos de Source. Si new_rows=0, no se escribe ningun archivo.


In [ ]:
try:
    report = BronzeIngestor(spark, config_path).run(TABLE, source_path=source_path, mode=LOAD_MODE)
    print(report)
finally:
    # Libera los workers tambien si la validacion o la escritura fallan.
    spark.stop()
